# 04 · Detailed model study, frozen holdout, artifact, MLflow and registry

This is the final local modeling notebook. It consumes the untuned Phase-A comparison from notebook 03, runs a larger chronological study and bounded tuning inside training, freezes the validation-selected winner, evaluates that unchanged winner on holdout once, saves the complete `FraudPipeline`, verifies local prediction equivalence, produces diagnostics, and optionally logs/registers the same artifact in MLflow.

By default MLflow is **off** so the core notebook can run without the optional dependency. Set `FRAUD_NOTEBOOK_TRACK=1` before starting the kernel to enable it.

The detailed study retains GPU device settings from the baseline comparison and the saved feature-selection recipe.


## 1. Load the checked prepared data, recipe and baseline comparison


In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Locate the repository from either the project root or notebook/.
current_directory = Path.cwd().resolve()
PROJECT_ROOT = None
for directory in [current_directory, *current_directory.parents]:
    if (directory / "src" / "fraud_detection").is_dir():
        PROJECT_ROOT = directory
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project root. Expected src/fraud_detection."
    )

source_path = str(PROJECT_ROOT / "src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)

SEED = 42
SMOKE = os.environ.get("FRAUD_NOTEBOOK_SMOKE", "0") == "1"
MAX_ROWS = int(os.environ.get("FRAUD_NOTEBOOK_MAX_ROWS", "0"))
if MAX_ROWS < 0:
    raise ValueError("FRAUD_NOTEBOOK_MAX_ROWS must be nonnegative.")

stage_setting = os.environ.get(
    "FRAUD_NOTEBOOK_STAGE_DIR", "data/processed/learning_stages"
)
STAGE_ROOT = Path(stage_setting).expanduser()
if not STAGE_ROOT.is_absolute():
    STAGE_ROOT = PROJECT_ROOT / STAGE_ROOT
STAGE_ROOT = STAGE_ROOT.resolve()
STAGE_ROOT.mkdir(parents=True, exist_ok=True)

print({
    "project_root": str(PROJECT_ROOT),
    "stage_root": str(STAGE_ROOT),
    "smoke": SMOKE,
    "max_rows": MAX_ROWS,
    "seed": SEED,
})

from fraud_detection.data_preparation import load_prepared_data
from fraud_detection.data import calculate_file_sha256
from fraud_detection.utils import frame_fingerprint

pointer_path = STAGE_ROOT / "prepared_latest.json"
if not pointer_path.is_file():
    raise FileNotFoundError(
        f"Missing {pointer_path}. Run notebook 00_data_preparation.ipynb first."
    )
pointer = json.loads(pointer_path.read_text(encoding="utf-8"))

if bool(pointer.get("smoke")) != SMOKE:
    raise ValueError(
        "Notebook mode differs from notebook 00. Use the same FRAUD_NOTEBOOK_SMOKE setting."
    )
if int(pointer.get("max_rows", 0)) != MAX_ROWS:
    raise ValueError(
        "FRAUD_NOTEBOOK_MAX_ROWS differs from notebook 00. Use the same row limit."
    )

prepared_folder = Path(pointer["prepared_folder"])
prepared_data = load_prepared_data(prepared_folder)
manifest_path = prepared_folder / "prepared_manifest.json"
actual_manifest_sha256 = calculate_file_sha256(manifest_path)
if actual_manifest_sha256 != pointer["prepared_manifest_sha256"]:
    raise ValueError("Prepared manifest changed since notebook 00.")

partitions = prepared_data["partitions"]
labeled_pool = prepared_data["labeled_pool"]
kaggle_test = prepared_data["kaggle_test"]
DATA_SCOPE = prepared_data["manifest"]["data_scope"]
if DATA_SCOPE != pointer["data_scope"]:
    raise ValueError("Prepared data scope differs from notebook 00 pointer.")

current_fingerprints = {
    name: frame_fingerprint(getattr(partitions, name))
    for name in ("train", "validation", "holdout")
}
if current_fingerprints != pointer["partition_fingerprints"]:
    raise ValueError("Prepared partition fingerprints differ from notebook 00.")

SOURCE_FILE_SHA256 = dict(pointer.get("source_file_sha256", {}))
JOIN_QUALITY_REPORT = prepared_data.get("quality_report", {})

print({
    "data_scope": DATA_SCOPE,
    "prepared_folder": str(prepared_folder),
    "train_rows": len(partitions.train),
    "validation_rows": len(partitions.validation),
    "holdout_rows": len(partitions.holdout),
    "kaggle_test_rows": len(kaggle_test),
    "integrity": "checksum + shape + partition fingerprints verified",
})

from fraud_detection.experiments import (
    evaluate_selected_comparison,
    load_comparison,
    save_comparison,
    staged_model_study,
    validate_comparison_contract,
)
from fraud_detection.models import save_results
from fraud_detection.predict import FraudPredictor

model_recipe_path = STAGE_ROOT / "model_recipe.json"
baseline_pointer_path = STAGE_ROOT / "baseline_latest.json"
if not model_recipe_path.is_file() or not baseline_pointer_path.is_file():
    raise FileNotFoundError("Run notebooks 02 and 03 first.")
model_recipe = json.loads(model_recipe_path.read_text(encoding="utf-8"))
# Earlier recipes used CPU selection before this field was introduced.
model_recipe.setdefault("selection_device", "cpu")
baseline_pointer = json.loads(baseline_pointer_path.read_text(encoding="utf-8"))
if baseline_pointer["prepared_manifest_sha256"] != pointer["prepared_manifest_sha256"]:
    raise ValueError("Baseline comparison belongs to a different prepared dataset.")

baseline = load_comparison(baseline_pointer["comparison_folder"])
validate_comparison_contract(baseline, partitions, DATA_SCOPE)
assert not baseline.metadata["holdout_evaluated"]
assert all("holdout" not in r.metrics for r in baseline.results.values())
display(baseline.table)


## 2. Run the detailed chronological study and freeze the winner


In [ ]:
SCREENING_FOLDS = int(baseline.metadata["n_splits"])
DETAIL_FOLDS = int(os.environ.get("FRAUD_NOTEBOOK_DETAIL_FOLDS", str(max(3, SCREENING_FOLDS + 1))))
DETAIL_ESTIMATORS = int(os.environ.get("FRAUD_NOTEBOOK_DETAIL_ESTIMATORS", "12" if SMOKE else "160"))
SHORTLIST_SIZE = min(
    int(os.environ.get("FRAUD_NOTEBOOK_SHORTLIST_SIZE", "3")),
    len(baseline.results),
)
TUNE = os.environ.get("FRAUD_NOTEBOOK_TUNE", "0" if SMOKE else "1") == "1"
INCLUDE_ENSEMBLE = os.environ.get("FRAUD_NOTEBOOK_ENSEMBLE", "0" if SMOKE else "1") == "1"

comparison_result = staged_model_study(
    partitions,
    screening_comparison=baseline,
    seed=SEED,
    screening_folds=SCREENING_FOLDS,
    detail_folds=DETAIL_FOLDS,
    shortlist_size=SHORTLIST_SIZE,
    detail_estimators=DETAIL_ESTIMATORS,
    tune=TUNE,
    data_scope=DATA_SCOPE,
    include_ensemble=INCLUDE_ENSEMBLE,
)
comparison_result.metadata.update(
    feature_recipe={
        key: model_recipe[key]
        for key in (
            "encoding", "selection_method", "selection_device", "selection_max_features",
            "feature_groups", "n_clusters", "raw_screen_strategy",
            "top_v_features", "preserve_features",
            "selection_consensus_components",
        )
    },
    prepared_manifest_sha256=pointer["prepared_manifest_sha256"],
    source_file_sha256=SOURCE_FILE_SHA256,
    join_quality_report=JOIN_QUALITY_REPORT,
)
for candidate_result in comparison_result.results.values():
    candidate_result.metadata.update(
        source_file_sha256=SOURCE_FILE_SHA256,
        join_quality_report=JOIN_QUALITY_REPORT,
    )

display(comparison_result.table)
print({
    "shortlist": comparison_result.metadata["shortlist"],
    "winner": comparison_result.winner_name,
    "holdout_evaluated": comparison_result.metadata["holdout_evaluated"],
    "ensemble_policy": comparison_result.metadata["ensemble_policy"],
})
assert not comparison_result.metadata["holdout_evaluated"]


## 3. Persist the frozen pre-holdout handoff, then assess once


In [ ]:
comparison_root = STAGE_ROOT / "model_comparisons"
comparison_root.mkdir(parents=True, exist_ok=True)
frozen_folder = save_comparison(comparison_result, output_dir=comparison_root)

# From this line onward, no fitting or threshold selection is allowed.
result = evaluate_selected_comparison(comparison_result, partitions)
assert result is comparison_result.winner
assert comparison_result.metadata["holdout_evaluated"]

assessed_folder = save_comparison(comparison_result, output_dir=comparison_root)
final_pointer = {
    "schema_version": 1,
    "stage": "final_assessed",
    "frozen_comparison_folder": str(frozen_folder.resolve()),
    "assessed_comparison_folder": str(assessed_folder.resolve()),
    "comparison_id": comparison_result.metadata["comparison_id"],
    "winner_name": comparison_result.winner_name,
    "winner_artifact_id": result.metadata["artifact_id"],
    "data_scope": DATA_SCOPE,
    "threshold": result.threshold,
}
final_pointer_path = STAGE_ROOT / "final_latest.json"
final_pointer_path.write_text(json.dumps(final_pointer, indent=2), encoding="utf-8")

display(pd.DataFrame(result.metrics).T)


## 4. Save the complete raw-row model and verify local inference


In [ ]:
model_root = PROJECT_ROOT / "models" / "local"
artifact_path = save_results(result, output_dir=model_root)
final_pointer["saved_pipeline_path"] = str(artifact_path.resolve())
final_pointer_path.write_text(json.dumps(final_pointer, indent=2), encoding="utf-8")

inputs = result.validation_inputs.iloc[:8].copy()
expected_probabilities = result.pipeline.predict_proba(inputs)[:, 1]
expected_labels = result.pipeline.predict(inputs)

predictor = FraudPredictor(model_path=artifact_path).load_model()
actual_probabilities = predictor.predict(inputs, return_probabilities=True)
actual_labels = predictor.predict(inputs, return_probabilities=False)
np.testing.assert_allclose(actual_probabilities, expected_probabilities, rtol=0, atol=1e-12)
np.testing.assert_array_equal(actual_labels, expected_labels)
print({"pipeline_path": str(artifact_path), "save_load_equivalence": "passed"})


## 5. Generate evaluation reports and fitted-model explanations


In [ ]:
from fraud_detection.evaluation import (
    candidate_feature_importance,
    error_analysis,
    model_card,
    save_evaluation_reports,
)

report_paths = save_evaluation_reports(
    result,
    partitions,
    artifact_path.parent,
    data_scope=DATA_SCOPE,
)
card = model_card(result, partitions, data_scope=DATA_SCOPE)
display(pd.Series({
    "artifact_id": card["artifact_id"],
    "data_scope": card["data_scope"],
    "model": card["model"],
    "encoding": card["encoding"],
    "selection_method": card["selection_method"],
    "threshold": card["threshold"],
    "validation_status": card["validation_status"],
    "production_ready": card["production_ready"],
}, name="model_card_summary"))
display(error_analysis(result, partitions))
display(candidate_feature_importance(result).head(20))
print(report_paths)


## 6. Score the complete unlabeled IEEE-CIS test table


In [ ]:
prediction_dir = artifact_path.parent / "predictions"
prediction_dir.mkdir(parents=True, exist_ok=True)
full_prediction_path = prediction_dir / "unlabeled_predictions.csv"
if full_prediction_path.exists():
    # FraudPredictor protects existing outputs. Use a run-specific file on reruns.
    import uuid
    full_prediction_path = prediction_dir / f"unlabeled_predictions_{uuid.uuid4().hex[:8]}.csv"

full_predictions = predictor.predict_batch(kaggle_test, full_prediction_path)
np.testing.assert_array_equal(
    full_predictions.TransactionID.to_numpy(),
    kaggle_test.TransactionID.to_numpy(),
)
print({
    "rows_scored": len(full_predictions),
    "output": str(full_prediction_path.resolve()),
    "metrics": "not available because the Kaggle test table has no labels",
})
display(full_predictions.head())


## 7. Optional MLflow tracking and registry


In [ ]:
TRACK = os.environ.get("FRAUD_NOTEBOOK_TRACK", "0") == "1"
REGISTER = os.environ.get("FRAUD_NOTEBOOK_REGISTER", "1") == "1"

if not TRACK:
    tracking_info = {"status": "skipped", "reason": "FRAUD_NOTEBOOK_TRACK is not 1"}
    display(pd.Series(tracking_info))
else:
    from fraud_detection.tracking import (
        DEFAULT_REGISTERED_MODEL,
        log_comparison,
        log_result,
        verify_registered_lineage,
    )

    if SMOKE:
        smoke_store = PROJECT_ROOT / ".mlflow" / "notebook-smoke"
        smoke_store.mkdir(parents=True, exist_ok=True)
        tracking_uri = "sqlite:///" + (smoke_store / "mlflow.db").as_posix()
        experiment_name = "ieee_cis_fraud_detection_notebook_smoke"
        model_name = "ieee_cis_fraud_detector_smoke"
    else:
        tracking_uri = os.environ.get("FRAUD_NOTEBOOK_TRACKING_URI") or None
        experiment_name = os.environ.get(
            "FRAUD_NOTEBOOK_EXPERIMENT", "ieee_cis_fraud_detection"
        )
        model_name = os.environ.get(
            "FRAUD_NOTEBOOK_MODEL_NAME", DEFAULT_REGISTERED_MODEL
        )

    comparison_run_id = log_comparison(
        comparison_result,
        assessed_folder,
        tracking_uri=tracking_uri,
        experiment=experiment_name,
    )
    run_id = log_result(
        result=result,
        partitions=partitions,
        saved_pipeline_path=artifact_path,
        folder=artifact_path.parent,
        comparison=comparison_result,
        data_scope=DATA_SCOPE,
        tracking_uri=tracking_uri,
        experiment=experiment_name,
        register_name=model_name if REGISTER else None,
        register_stage="Staging" if REGISTER else None,
    )
    tracking_info = {
        "status": "logged",
        "tracking_uri": tracking_uri,
        "experiment": experiment_name,
        "comparison_run_id": comparison_run_id,
        "run_id": run_id,
        "registered_name": model_name if REGISTER else None,
        "registered_stage": "Staging" if REGISTER else None,
    }
    display(pd.Series(tracking_info))


## 8. If registered, verify registry lineage and prediction equivalence


In [ ]:
if TRACK and REGISTER:
    import mlflow

    # log_result configured the requested tracking store. Rebind explicitly for
    # a fresh/re-executed cell before querying the registry.
    from fraud_detection.tracking import configure_tracking
    resolved_tracking_uri = configure_tracking(tracking_info["tracking_uri"], tracking_info["experiment"])
    client = mlflow.tracking.MlflowClient()
    versions = [
        version for version in client.search_model_versions(f"name='{tracking_info['registered_name']}'")
        if version.run_id == tracking_info["run_id"]
    ]
    if not versions:
        raise RuntimeError("Could not find the model version registered by this MLflow run.")
    registered_version = max(versions, key=lambda v: int(v.version))
    registry_uri = f"models:/{tracking_info['registered_name']}/{registered_version.version}"

    verify_registered_lineage(
        tracking_info["run_id"],
        partitions,
        tracking_uri=resolved_tracking_uri,
    )
    registered_pipeline = mlflow.sklearn.load_model(registry_uri)
    registered_probabilities = registered_pipeline.predict_proba(inputs)[:, 1]
    registered_labels = registered_pipeline.predict(inputs)
    np.testing.assert_allclose(
        registered_probabilities, expected_probabilities, rtol=0, atol=1e-12
    )
    np.testing.assert_array_equal(registered_labels, expected_labels)

    registry_evidence = {
        "run_id": tracking_info["run_id"],
        "registered_model": tracking_info["registered_name"],
        "registered_version": str(registered_version.version),
        "registry_uri": registry_uri,
        "lineage_verified": True,
        "probabilities_verified": True,
        "labels_verified": True,
    }
    (STAGE_ROOT / "registry_verification.json").write_text(
        json.dumps(registry_evidence, indent=2), encoding="utf-8"
    )
    display(pd.Series(registry_evidence))
else:
    print("Registry verification skipped because tracking/registration is disabled.")


## Final state

At this point the local workflow has one frozen winner, one final holdout assessment, a saved raw-row `FraudPipeline`, reproducible lineage, an unlabeled-test prediction export, and—when enabled—the same model logged and registered in MLflow. Do not tune the model after inspecting these holdout metrics; a new decision requires a new final assessment period.


## 9. Inspect the saved LightGBM improvement experiment

Run the following cell after the model evaluation cell above. It fits one improved GPU LightGBM model using the already fitted feature pipeline, then compares it with the original model. It creates no extra files.

The comparison below uses the same 23,621 transactions in the last chronological validation block for both models. These are **development results**: the original model selection already used the full validation period. They do not replace the original holdout assessment. A new final assessment period is needed for a later model version.

The fitted objects stay in memory as `improved_model`, `improved_calibrator`, and `predict_improved`. The original `result`, `predictor`, and final-model pointer are left unchanged.


In [ ]:
if "result" not in globals() or "partitions" not in globals():
    raise RuntimeError("Run notebook 04 through the model evaluation cell first.")

from IPython.display import display
from lightgbm import LGBMClassifier, early_stopping
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, confusion_matrix,
    f1_score, precision_recall_curve, precision_score, recall_score,
    roc_auc_score,
)

# Reuse the transformations already fitted on the training partition.
feature_pipeline = result.pipeline.pipeline[:-1]
train_rows = partitions.train.drop(columns="isFraud")
validation_rows = partitions.validation.drop(columns="isFraud")
X_train = feature_pipeline.transform(train_rows)
X_validation = feature_pipeline.transform(validation_rows)
y_train = partitions.train.isFraud.to_numpy()
y_validation = partitions.validation.isFraud.to_numpy()

# Use the first 60% of validation for early stopping, the next 20%
# for calibration and threshold selection, and the last 20% for comparison.
tuning_end = int(len(y_validation) * 0.60)
calibration_end = int(len(y_validation) * 0.80)
assessment = slice(calibration_end, None)

improved_model = LGBMClassifier(
    n_estimators=600,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=30,
    reg_alpha=0.1,
    reg_lambda=5.0,
    colsample_bytree=0.9,
    subsample=0.9,
    subsample_freq=1,
    class_weight=None,
    device_type="gpu",
    random_state=SEED,
    n_jobs=2,
    verbosity=-1,
)
improved_model.fit(
    X_train,
    y_train,
    eval_set=[(X_validation.iloc[:tuning_end], y_validation[:tuning_end])],
    eval_metric="average_precision",
    callbacks=[early_stopping(60, first_metric_only=True, verbose=False)],
)

raw_scores = improved_model.predict_proba(X_validation)[:, 1]

def score_to_logit(scores):
    scores = np.clip(np.asarray(scores), 1e-6, 1 - 1e-6)
    return (np.log(scores) - np.log1p(-scores)).reshape(-1, 1)

improved_calibrator = LogisticRegression(C=1e6, max_iter=1000, random_state=SEED)
calibration_slice = slice(tuning_end, calibration_end)
improved_calibrator.fit(
    score_to_logit(raw_scores[calibration_slice]),
    y_validation[calibration_slice],
)
calibrated_scores = improved_calibrator.predict_proba(score_to_logit(raw_scores))[:, 1]

precision, recall, thresholds = precision_recall_curve(
    y_validation[calibration_slice], calibrated_scores[calibration_slice]
)
f1_values = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
best_threshold = int(np.flatnonzero(np.isclose(f1_values, f1_values.max()))[-1])
improved_threshold = float(thresholds[best_threshold])

def make_metrics(labels, scores, threshold):
    predicted = scores >= threshold
    tn, fp, fn, tp = confusion_matrix(labels, predicted, labels=[0, 1]).ravel()
    return {
        "roc_auc": roc_auc_score(labels, scores),
        "average_precision": average_precision_score(labels, scores),
        "precision": precision_score(labels, predicted, zero_division=0),
        "recall": recall_score(labels, predicted, zero_division=0),
        "f1": f1_score(labels, predicted, zero_division=0),
        "brier_score": brier_score_loss(labels, scores),
        "true_positive": tp, "false_positive": fp,
        "false_negative": fn, "rows": len(labels),
    }

original_classifier = result.pipeline.pipeline.named_steps["classifier"]
original_scores = original_classifier.predict_proba(X_validation)[:, 1]
comparison = pd.DataFrame([
    {"model": "Original", **make_metrics(y_validation[assessment], original_scores[assessment], result.threshold)},
    {"model": "Improved LightGBM + calibration", **make_metrics(y_validation[assessment], calibrated_scores[assessment], improved_threshold)},
]).set_index("model")
display(comparison)
display(pd.Series({
    "best_iteration": improved_model.best_iteration_,
    "threshold": improved_threshold,
    "tuning_rows": tuning_end,
    "calibration_rows": calibration_end - tuning_end,
    "assessment_rows": len(y_validation) - calibration_end,
}, name="improved_model_info"))

def predict_improved(raw_rows):
    features = feature_pipeline.transform(raw_rows)
    scores = improved_model.predict_proba(features)[:, 1]
    return improved_calibrator.predict_proba(score_to_logit(scores))[:, 1]

print("Finished in this notebook cell only. No model or report files were written.")
del X_train, X_validation, train_rows, validation_rows
